# JackTheLearner - Full Training Pipeline

**A general-purpose humanoid robot brain - 105M parameters**

## Training Philosophy
```
Phase 0: Physics Foundation     - Learn how the world works
Phase 1: Imitation (MoCap)      - Learn what human movement LOOKS like
Phase 2: Locomotion RL          - Make imitated walking WORK
Phase 3: Perception             - Learn to SEE and UNDERSTAND
Phase 4: Manipulation           - Connect vision to motor skills
Phase 5: Audio                  - Speech recognition + response
Phase 6: Planning               - Hierarchical task decomposition
Phase 7: Integration            - Everything together
```

## Key Design Decisions
- **Motor-first**: Learn to move before learning to understand
- **Component freezing**: Prevent catastrophic forgetting
- **Reinforcement loops**: Every component learns from action outcomes

## Research Backing
| Component | Paper |
|-----------|-------|
| AMP Discriminator | [Adversarial Motion Priors (Peng 2021)](https://arxiv.org/abs/2104.02180) |
| MoCap Imitation | [DeepMimic (Peng 2018)](https://arxiv.org/abs/1804.02717) |
| Terrain Curriculum | [Legged Gym (Rudin 2022)](https://arxiv.org/abs/2109.11978) |
| Vision-Action | [RT-2 (Brohan 2023)](https://arxiv.org/abs/2307.15818) |
| World Model | [TD-MPC2 (Hansen 2024)](https://arxiv.org/abs/2310.16828) |
| Dual System | [π₀ (Physical Intelligence 2024)](https://www.physicalintelligence.company/blog/pi0) |

---

## 1. Setup Environment

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PATH = '/content/drive/MyDrive/JackTheLearner/checkpoints'
os.makedirs(DRIVE_PATH, exist_ok=True)
print('Google Drive mounted!')
print(f'Checkpoints will be backed up to: {DRIVE_PATH}')

In [ ]:
# Install dependencies
!pip install -q torch torchvision
!pip install -q gymnasium mujoco
!pip install -q sympy tqdm transformers
print('Dependencies installed!')

In [ ]:
# Verify GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU! Go to Runtime > Change runtime type > T4/A100 GPU')

In [ ]:
# Clone repository
%cd /content
!rm -rf JackTheLearner 2>/dev/null
!git clone https://github.com/JannoLouwrens/JackTheLearner.git
%cd JackTheLearner

# Create directories
!mkdir -p checkpoints
!mkdir -p datasets/cmu_mocap
!mkdir -p datasets/cache

# Copy any existing checkpoints FROM Drive (for resume)
!cp /content/drive/MyDrive/JackTheLearner/checkpoints/*.pt checkpoints/ 2>/dev/null || echo 'No existing checkpoints on Drive'

print('\nRepository cloned!')

---
## 1.5 Download Datasets (Optional but Recommended)

**CMU MoCap Database**: Human motion capture data for Phase 1 imitation learning.
- Without this: Uses synthetic movement patterns (still works, less realistic)
- With this: Learns from real human movement recordings

**Pretrained Models**: Downloaded automatically by HuggingFace in Phase 3+
- DINOv2 (facebook/dinov2-large): ~1.2GB
- SigLIP (openai/clip-vit-large-patch14): ~900MB
- Whisper (openai/whisper-tiny): ~150MB (Phase 5)
- Phi-2/Qwen2 (LLM backbone): ~2-5GB (Phase 3+)

**Note**: Phase 0-2 work WITHOUT any downloads (physics + synthetic motion)

In [ ]:
# Download CMU MoCap data (OPTIONAL - improves Phase 1 quality)
# This downloads a selection of locomotion, reaching, and manipulation clips

import os

MOCAP_DIR = 'datasets/cmu_mocap'
os.makedirs(MOCAP_DIR, exist_ok=True)

# CMU MoCap subject/trial files (BVH format)
# Subject 01-09: General locomotion
# Subject 13-15: Walking variations
# Subject 35: Running
# Subject 49: Jumping
# Subject 55-56: Object manipulation
# Subject 61: Arm movements

MOCAP_FILES = [
    # Locomotion (walking, running)
    ('01', '01'),  # Walk
    ('01', '02'),  # Walk
    ('02', '01'),  # Walk
    ('02', '02'),  # Walk
    ('07', '01'),  # Walk slow
    ('07', '02'),  # Walk
    ('08', '01'),  # Walk
    ('09', '01'),  # Run
    ('09', '02'),  # Run
    ('35', '01'),  # Run
    ('35', '02'),  # Run
    
    # Upper body / reaching
    ('13', '01'),  # Walk with arm swing
    ('13', '17'),  # Arm movements
    ('14', '01'),  # Walk
    ('15', '01'),  # Walk
    
    # Manipulation / grasping
    ('55', '01'),  # Object pick up
    ('55', '02'),  # Object manipulation
    ('56', '01'),  # Object interaction
    
    # Arm movements
    ('61', '01'),  # Arms
    ('61', '02'),  # Arms
]

print(f'Downloading {len(MOCAP_FILES)} MoCap files from CMU database...')

for subject, trial in MOCAP_FILES:
    filename = f'{subject}_{trial}.bvh'
    filepath = os.path.join(MOCAP_DIR, filename)
    
    if not os.path.exists(filepath):
        url = f'http://mocap.cs.cmu.edu/subjects/{subject}/{subject}_{trial}.bvh'
        !wget -q --timeout=30 {url} -O {filepath} 2>/dev/null || echo "Skip: {filename}"
    
print(f'\nMoCap files downloaded to {MOCAP_DIR}/')
!ls -la {MOCAP_DIR}/ | head -20

In [ ]:
# Pre-download HuggingFace models (OPTIONAL - saves time during Phase 3+)
# Skip this cell if you want models to download during training

PREDOWNLOAD_MODELS = False  # Set to True to download ~4GB of models now

if PREDOWNLOAD_MODELS:
    print('Pre-downloading HuggingFace models...')
    print('This will download ~4GB and take 5-10 minutes on Colab')
    
    from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM
    
    # Vision models (Phase 3)
    print('\n[1/4] DINOv2...')
    AutoModel.from_pretrained("facebook/dinov2-large")
    
    print('[2/4] SigLIP/CLIP...')
    AutoModel.from_pretrained("openai/clip-vit-large-patch14")
    
    # LLM backbone (Phase 3+) - using smaller model for Colab
    print('[3/4] Phi-2 (small LLM)...')
    AutoModelForCausalLM.from_pretrained("microsoft/phi-2", trust_remote_code=True)
    AutoTokenizer.from_pretrained("microsoft/phi-2", trust_remote_code=True)
    
    # Audio (Phase 5)
    print('[4/4] Whisper-tiny...')
    from transformers import WhisperProcessor, WhisperForConditionalGeneration
    WhisperProcessor.from_pretrained("openai/whisper-tiny")
    WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny")
    
    print('\n✓ All models cached! Phase 3+ will start faster.')
else:
    print('Skipping model pre-download.')
    print('Models will be downloaded automatically when needed (Phase 3+).')
    print('Set PREDOWNLOAD_MODELS = True above to download now.')

In [ ]:
# Verify setup
print('='*60)
print('SETUP VERIFICATION')
print('='*60)

import os
import torch

# Check GPU
print(f'\n[GPU] CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'[GPU] Device: {torch.cuda.get_device_name(0)}')
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'[GPU] Memory: {mem_gb:.1f} GB')
    if mem_gb < 15:
        print('[WARN] <16GB VRAM - Phase 3+ may need smaller batch sizes')
    if mem_gb >= 40:
        print('[OK] A100 detected - full training supported!')

# Check MoCap
mocap_files = os.listdir('datasets/cmu_mocap') if os.path.exists('datasets/cmu_mocap') else []
bvh_files = [f for f in mocap_files if f.endswith('.bvh')]
print(f'\n[MoCap] BVH files: {len(bvh_files)}')
if len(bvh_files) == 0:
    print('[WARN] No MoCap data - Phase 1 will use synthetic fallbacks')
else:
    print(f'[OK] MoCap ready for imitation learning')

# Check checkpoints
ckpt_files = os.listdir('checkpoints') if os.path.exists('checkpoints') else []
pt_files = [f for f in ckpt_files if f.endswith('.pt')]
print(f'\n[Checkpoints] Found: {len(pt_files)}')
for f in sorted(pt_files)[:5]:
    print(f'  - {f}')
if len(pt_files) > 5:
    print(f'  ... and {len(pt_files)-5} more')

print('\n' + '='*60)
print('Setup complete! Ready to train.')
print('='*60)

---
## 2. Phase 0: Physics Foundation (15-30 min)

**Goal**: Learn how the physical world works

**What trains**: State encoder, physics predictor

**Method**: Predict next state from (state, action) pairs using SymPy ground truth

In [ ]:
# Phase 0: Physics Foundation
!python RobustTrainer.py --phase 0 --epochs 50
print('\nPhase 0 complete! Physics knowledge learned.')

---
## 3. Phase 1: Imitation Learning (2-4 hours)

**Goal**: Learn what human movement LOOKS like from MoCap data

**Subphases**:
- 1.1: Locomotion (walking, running, turning)
- 1.2: Upper body (reaching, arm movements)
- 1.3: Manipulation (grasping, finger movements)
- 1.4: Combined (full body coordination)

**Key components**:
- Behavior Cloning: Copy MoCap joint angles
- AMP Discriminator: Ensure motion looks human-like
- Smoothness loss: Penalize jerky motion
- Physics validation: Check robot doesn't fall

**What's FROZEN**: Vision, LLM, Object Detection (perception not needed yet)

In [ ]:
# Phase 1: Imitation Learning from MoCap
!python RobustTrainer.py --phase 1 --epochs 200
print('\nPhase 1 complete! Human-like movement learned.')

---
## 4. Phase 2: Locomotion RL (2-4 hours)

**Goal**: Make imitated walking actually WORK in varied conditions

**Subphases**:
- 2.1: Walking RL on flat ground (refine with physics feedback)
- 2.2: Terrain adaptation (stairs, slopes, rough)
- 2.3: Domain randomization (mass ±20%, friction ±30%)

**Key**: MoCap prior prevents ugly motions, RL makes it robust

In [ ]:
# Phase 2: Locomotion RL
!python RobustTrainer.py --phase 2 --epochs 300
print('\nPhase 2 complete! Robust walking learned.')

---
## 5. Phase 3: Perception (2-4 hours)

**Goal**: Learn to SEE and UNDERSTAND

**Subphases**:
- 3.1: Vision Training (DINOv2 + SigLIP) - with action feedback!
- 3.2: Object Detection - verified by grasp success
- 3.3: LLM Projector - verified by execution feedback
- 3.4: Language-Vision Grounding - connect words to visual objects

**What's FROZEN**: Motor skills (action_head, proprio) - preserve Phase 1-2 learning

**Requires A100 GPU** for vision models!

In [ ]:
# Phase 3: Perception (Vision + LLM)
!python RobustTrainer.py --phase 3 --epochs 200
print('\nPhase 3 complete! Robot can see and understand.')

---
## 6. Phase 4: Vision-Guided Manipulation (4-8 hours)

**Goal**: Connect perception to motor skills

**Subphases**:
- 4.1: Vision-Guided Reaching - "reach for the cup"
- 4.2: Vision-Guided Grasping - "grasp the bottle"
- 4.3: Loco-Manipulation - "bring the cup to the counter"

**Full reinforcement loop**: Command → Vision → Detection → Motor → Feedback → Update ALL

In [ ]:
# Phase 4: Vision-Guided Manipulation
!python RobustTrainer.py --phase 4 --epochs 300
print('\nPhase 4 complete! Vision-guided manipulation learned.')

---
## 7. Phase 5: Audio Integration (1-2 hours)

**Goal**: Speech recognition + response

**Subphases**:
- 5.1: Speech Recognition (Whisper-style)
- 5.2: Speech Response (TTS feedback)

In [ ]:
# Phase 5: Audio Integration
!python RobustTrainer.py --phase 5 --epochs 150
print('\nPhase 5 complete! Speech capabilities added.')

---
## 8. Phase 6: Advanced Planning (2-4 hours)

**Goal**: Complex task decomposition, world modeling, navigation

**Subphases**:
- 6.1: Hierarchical Planning - "make coffee" → [subtasks]
- 6.2: World Model (TD-MPC2) - predict before acting
- 6.3: Navigation Planning - path planning with obstacles

In [ ]:
# Phase 6: Advanced Planning
!python RobustTrainer.py --phase 6 --epochs 200
print('\nPhase 6 complete! Planning capabilities added.')

---
## 9. Phase 7: Full Integration (4-8 hours)

**Goal**: All systems working together at different timescales

**Dual System Architecture** (inspired by π₀):
- System 2 (2-5 Hz): High-level planning
- System 1 (10-20 Hz): Action chunk generation
- System 0 (500 Hz): Low-level motor control

**End-to-end tasks**: "Make me coffee and bring it here"

In [ ]:
# Phase 7: Full Integration
!python RobustTrainer.py --phase 7 --epochs 300
print('\nPhase 7 complete! Full integration achieved.')
print('\n' + '='*70)
print('TRAINING COMPLETE!')
print('='*70)

---
## 10. Backup & Verify

In [ ]:
# List all checkpoints
print('LOCAL checkpoints:')
!ls -lh checkpoints/
print('\nDRIVE checkpoints:')
!ls -lh /content/drive/MyDrive/JackTheLearner/checkpoints/

In [ ]:
# Manual backup to Drive
import shutil, glob, os
src = '/content/JackTheLearner/checkpoints'
dst = '/content/drive/MyDrive/JackTheLearner/checkpoints'
for f in glob.glob(f'{src}/*.pt'):
    shutil.copy2(f, dst)
    print(f'Copied {os.path.basename(f)}')
print('Backup complete!')

---
## 11. Test the Trained Model

In [ ]:
# Run integration test
!python test_integration.py

---
## Checkpoint Reference

| Phase | Checkpoint | Description |
|-------|------------|-------------|
| 0 | `phase0_best.pt` | Physics knowledge |
| 1 | `phase1_best.pt` | Imitation (MoCap) |
| 1 | `phase1_1_best.pt` | Locomotion imitation |
| 1 | `phase1_2_best.pt` | Upper body imitation |
| 1 | `phase1_3_best.pt` | Manipulation imitation |
| 1 | `phase1_4_best.pt` | Combined imitation |
| 2 | `phase2_best.pt` | Locomotion RL |
| 3 | `phase3_best.pt` | Perception |
| 4 | `phase4_best.pt` | Manipulation |
| 5 | `phase5_complete.pt` | Audio |
| 6 | `phase6_complete.pt` | Planning |
| 7 | `phase7_complete.pt` | Full Integration |
| 7 | `final_model.pt` | Final trained model |

---
**Model**: UnifiedBrain - 105M parameters (~420MB)  
**Author**: Janno Louwrens